## 6. Results summary

| Recipe | 5-fold cv_mae | Δ vs baseline |
|--------|--------------:|--------------:|
| Ridge baseline (3 features) | 96,693.85 | — |
| HGB squared_error, all numeric+bool | 70,431.74 | −27.2% |
| HGB absolute_error, same features | 56,069.80 | −42.0% |
| + native cats (commune_codes, property_type) | 52,355.69 | −45.9% |
| **+ log1p target + ratio features + bigger HGB** | **50,934.78** | **−47.3%** |

The single biggest lever was switching to `loss="absolute_error"` (−14k MAE). The final 1.4k drop on top of the prior winner came from combining a log-transformed target with structural ratio features and a higher-capacity HGB (more leaves, more iterations, lower learning rate) — the smaller capacity used by earlier iters was the binding constraint, and the log target made the residuals less heteroskedastic.

In [ ]:
X_test = add_ratio_features(test_df).reindex(columns=feature_columns)

final_pipe = build_pipeline(numeric_features)
final_pipe.fit(X_full, np.log1p(y_full))
test_preds = np.expm1(final_pipe.predict(X_test))

assert len(test_preds) == len(test_df)
assert not np.isnan(test_preds).any()
assert not np.isinf(test_preds).any()
assert (test_preds >= 0).all()

submission = [{"property_value": float(v)} for v in test_preds]
out_json = ROOT / "baseline" / "predicted.json"
out_zip = ROOT / "baseline" / "predicted.zip"
out_json.parent.mkdir(parents=True, exist_ok=True)
with open(out_json, "w") as f:
    json.dump(submission, f)
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(out_json, arcname="predicted.json")

print(f"wrote {len(submission)} predictions -> {out_json}")
print(f"wrote zip                -> {out_zip}")
print(f"median pred: {np.median(test_preds):,.2f}  median train: {np.median(y_full):,.2f}")

## 5. Refit on full train + write submission

Refit one pipeline on the full training set and predict the test rows. The submission file format expected by the grader is a JSON list of `{"property_value": <float>}` records, in the same row order as `dataset/test.json`, plus a zip with the same JSON inside under entry name `predicted.json`.

In [ ]:
def build_pipeline(numeric_features):
    numeric_pipe = SimpleImputer(strategy="median")
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, CATEGORICAL_FEATURES),
    ])
    cat_mask = [False] * len(numeric_features) + [True] * len(CATEGORICAL_FEATURES)
    model = HistGradientBoostingRegressor(
        loss="absolute_error",
        max_iter=1500,
        learning_rate=0.02,
        max_leaf_nodes=127,
        min_samples_leaf=20,
        l2_regularization=0.0,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=30,
        categorical_features=cat_mask,
    )
    return Pipeline([("preprocessor", preprocessor), ("model", model)])


X_full = add_ratio_features(train_df).reindex(columns=feature_columns)
y_full = train_df[TARGET_KEY].to_numpy(dtype=np.float64)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
fold_maes = []
for i, (tr, va) in enumerate(kf.split(X_full)):
    pipe = build_pipeline(numeric_features)
    pipe.fit(X_full.iloc[tr], np.log1p(y_full[tr]))
    pred_va = np.expm1(pipe.predict(X_full.iloc[va]))
    mae = mean_absolute_error(y_full[va], pred_va)
    fold_maes.append(mae)
    print(f"fold_{i}_mae: {mae:>13,.2f}")

cv_mae = float(np.mean(fold_maes))
cv_mae_std = float(np.std(fold_maes))
print(f"\ncv_mae:     {cv_mae:>13,.2f}")
print(f"cv_mae_std: {cv_mae_std:>13,.2f}")

## 4. Model + 5-fold CV-MAE

The model is a `HistGradientBoostingRegressor` with:
- **`loss="absolute_error"`** — directly optimizes MAE on residuals (median-fitting).
- **log-transformed target** (`np.log1p`, predictions back-transformed via `np.expm1`) — compresses heavy tails so the same loss function sees a more balanced distribution.
- **High-capacity tree settings** — `max_leaf_nodes=127`, `max_iter=1500`, `learning_rate=0.02`, `min_samples_leaf=20`, `l2_regularization=0.0`. Early stopping with `n_iter_no_change=30` and `validation_fraction=0.15` lets the optimizer find its own iteration count.
- **Native categorical handling** for `commune_codes` and `property_type` (no one-hot blow-up).

5-fold CV is run with `KFold(shuffle=True, random_state=42)` exactly as the locked harness does — so the cell below should print `cv_mae ≈ 50,935`.

In [ ]:
TARGET_KEY = "property_value"
CATEGORICAL_FEATURES = ["commune_codes", "property_type"]

ROOM_COUNT_COLS = [
    "num_apt_1_room", "num_apt_2_rooms", "num_apt_3_rooms",
    "num_apt_4_rooms", "num_apt_5plus_rooms",
    "num_house_1_room", "num_house_2_rooms", "num_house_3_rooms",
    "num_house_4_rooms", "num_house_5plus_rooms",
]


def add_ratio_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rooms = df.reindex(columns=ROOM_COUNT_COLS).fillna(0).sum(axis=1)
    df["total_rooms"] = rooms
    built = df.get("built_area", pd.Series(np.nan, index=df.index))
    land = df.get("land_area", pd.Series(np.nan, index=df.index))
    premises = df.get("num_premises", pd.Series(np.nan, index=df.index))
    houses = df.get("num_houses", pd.Series(np.nan, index=df.index))
    apts = df.get("num_apartments", pd.Series(np.nan, index=df.index))
    df["built_per_room"] = built / (rooms + 1.0)
    df["area_per_lot"] = built / (land.fillna(0) + 1.0)
    df["house_share"] = houses / (premises.fillna(0) + 1.0)
    df["apt_share"] = apts / (premises.fillna(0) + 1.0)
    return df


def select_numeric_features(records):
    df = add_ratio_features(pd.DataFrame(records))
    numeric = df.select_dtypes(include=["number", "bool"]).columns.tolist()
    return [c for c in numeric if c != TARGET_KEY]


numeric_features = select_numeric_features(train_records)
feature_columns = numeric_features + CATEGORICAL_FEATURES
print(f"{len(numeric_features)} numeric/bool features + {len(CATEGORICAL_FEATURES)} cats = {len(feature_columns)} total")
print("ratio cols added:", [c for c in numeric_features if c in
    ("total_rooms", "built_per_room", "area_per_lot", "house_share", "apt_share")])

## 3. Feature engineering — winning recipe

The numeric/bool columns are used as-is (HGB tolerates NaNs natively but we still impute medians for clean preprocessing). On top, **structural ratio features** are computed without target leakage — they capture density / composition of the property:

- `total_rooms` — sum of all per-room counts (apartments + houses, all sizes).
- `built_per_room` — `built_area / (total_rooms + 1)`: average size per room.
- `area_per_lot` — `built_area / (land_area + 1)`: how built-up the parcel is.
- `house_share`, `apt_share` — fraction of premises that are houses / apartments.

`commune_codes` and `property_type` are treated as **native HGB categoricals** via OrdinalEncoder so the tree splits handle them directly.

In [ ]:
y = train_df["property_value"]
print("Target distribution (€):")
print(y.describe(percentiles=[0.05, 0.5, 0.95]).round(2).to_string())

print("\nMissingness top-10 columns:")
miss = train_df.isna().mean().sort_values(ascending=False)
print(miss.head(10).round(3).to_string())

print("\nKey categorical cardinalities:")
for c in ["commune_codes", "property_type", "transaction_type", "dept_code", "region_name"]:
    print(f"  {c:<20s} nunique={train_df[c].nunique()}")

## 2. Quick EDA

Three observations that drove the modeling choices:
- **Target is heavy-tailed** — median ~96,000 € but max in the millions. Squared-error loss pulls fits toward the mean and is suboptimal for MAE; an L1/`absolute_error` loss is the right starting point, and a log-target transform compresses the tail further.
- **High-cardinality spatial categorical** — `commune_codes` has 196 unique values (and is highly predictive of price), so it goes into the model as a native HGB categorical, not one-hot.
- **Most other categoricals are near-constants** — `dept_code`, `region_name`, `region_code`, `dept_name` each have only 1 unique value in training, so they carry no signal. The only other useful cat is `property_type` (25 values).

In [ ]:
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = Path.cwd()
TRAIN_PATH = ROOT / "dataset" / "train.json"
TEST_PATH = ROOT / "dataset" / "test.json"

train_records = json.load(open(TRAIN_PATH))
test_records = json.load(open(TEST_PATH))

train_df = pd.DataFrame(train_records)
test_df = pd.DataFrame(test_records)

print(f"train: {train_df.shape}, test: {test_df.shape}")
print(f"target dtype: {train_df['property_value'].dtype}")

## 1. Setup & data load

# Property-Value Prediction — Solution Notebook

**Task:** predict `property_value` (€) for 6,843 French real-estate transactions in `dataset/test.json`.
**Metric:** MAE on `property_value`.
**Baseline (Ridge on 3 features):** cv_mae ≈ 96,694
**Final (this notebook):** cv_mae ≈ **50,935** — a 47% reduction.

The recipe was discovered by an autonomous keep-or-revert search loop (`auto-mae`) over `experiment.py` against a locked 5-fold CV-MAE harness (`.claude/skills/auto-mae/eval.py`). This notebook reproduces the winning configuration end-to-end.